In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import load_dataset, Audio

dataset = load_dataset("PolyAI/minds14", "en-US", split="train")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset[11]

{'path': 'en-US~JOINT_ACCOUNT/602baf4fbb1e6d0fbce922b8.wav',
 'audio': {'path': '602baf4fbb1e6d0fbce922b8.wav',
  'array': array([ 0.        ,  0.        ,  0.        , ...,  0.        ,
         -0.00048828,  0.00024414]),
  'sampling_rate': 8000},
 'transcription': 'how do I go about setting up a joint account for my wife and I',
 'english_transcription': 'how do I go about setting up a joint account for my wife and I',
 'intent_class': 11,
 'lang_id': 4}

In [4]:
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

model = AutoModelForAudioClassification.from_pretrained("facebook/wav2vec2-base")
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

/usr/local/lib/python3.11/site-packages/transformers/configuration_utils.py:334: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
dataset[0]

{'path': 'en-US~JOINT_ACCOUNT/602ba55abb1e6d0fbce92065.wav',
 'audio': {'path': '602ba55abb1e6d0fbce92065.wav',
  'array': array([ 1.70562416e-05,  2.18727451e-04,  2.28099874e-04, ...,
          3.43842403e-05, -5.96364771e-06, -1.76846661e-05]),
  'sampling_rate': 16000},
 'transcription': 'I would like to set up a joint account with my partner',
 'english_transcription': 'I would like to set up a joint account with my partner',
 'intent_class': 11,
 'lang_id': 4}

In [6]:
def preprocess_function(examples):
    audio_arrays = [x['array'] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=True,
        max_length=100000,
        truncation=True,
    )
    return inputs

dataset = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 563/563 [01:06<00:00,  8.44 examples/s]


In [7]:
dataset = dataset.rename_column("intent_class", "labels")

In [8]:
from torch.utils.data import DataLoader

dataset.set_format(type="torch", columns=["input_values", "labels"])
dataloader = DataLoader(dataset, batch_size=4)

In [9]:
for data in dataloader:
    print(data)

{'labels': tensor([11, 11, 11, 11]), 'input_values': tensor([[ 3.7660e-04,  2.8342e-03,  2.9484e-03,  ..., -6.5770e-04,
          2.7497e-03,  4.6367e-03],
        [ 9.8650e-05,  3.7332e-03,  6.9454e-03,  ...,  1.3290e-02,
          1.8930e-02,  1.9547e-02],
        [ 2.2214e-04,  5.0336e-04,  2.9262e-04,  ..., -2.6319e+00,
         -2.1793e+00, -1.7696e+00],
        [ 2.8104e-03,  1.9075e-03,  2.7828e-04,  ..., -1.3702e-05,
         -1.3702e-05, -1.3702e-05]])}
{'labels': tensor([11, 11, 11, 11]), 'input_values': tensor([[-2.2645e-03, -1.0360e-03,  4.2160e-06,  ...,  1.3076e-05,
          1.3076e-05,  1.3076e-05],
        [ 1.0420e-03,  2.4938e-03,  4.1041e-03,  ...,  9.5005e-04,
          9.5005e-04,  9.5005e-04],
        [-7.0523e-03, -5.7372e-03,  9.9892e-04,  ...,  1.6357e-03,
          1.6357e-03,  1.6357e-03],
        [-5.3158e-04, -8.6423e-04,  2.6347e-04,  ..., -1.4535e-01,
         -3.1664e-01, -4.4494e-01]])}
{'labels': tensor([11, 11, 11, 11]), 'input_values': tensor([[-1.0